# Buổi 4 — Hồi quy tuyến tính & Phân tích nhân tố (Bài 10 + Bài 2 phần EFA)

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

In [ ]:
df.loc[df.study_hours > 20, "study_hours"] /= 10
df["interest"] = df[["h1", "h2", "h3"]].mean(axis=1); df["anxiety"] = df[["a1", "a2", "a3"]].mean(axis=1)
import statsmodels.formula.api as smf

## 1. Hồi quy đơn từ nguyên lý đầu tiên
Chọn đường thẳng làm **tổng bình phương phần dư nhỏ nhất**: β1 = cov(x,y)/var(x), β0 = ȳ − β1·x̄.
SPSS: `Analyze > Regression > Linear`.

In [ ]:
x, y = df.study_hours, df.math
b1 = np.cov(x, y)[0, 1] / x.var(); b0 = y.mean() - b1 * x.mean(); print(f"β1={b1:.3f}, β0={b0:.2f}")
m1 = smf.ols("math ~ study_hours", df).fit(); print(m1.params.round(3).to_dict(), "R²=", round(m1.rsquared, 3))

**❓** Diễn giải β1 bằng lời (đơn vị!). R² = bao nhiêu nghĩa là gì? Vì sao R² của hồi quy đơn = r² của Pearson?

## 2. Hồi quy đa biến

In [ ]:
m2 = smf.ols("math ~ study_hours + interest + anxiety + C(school)", df).fit()
print(m2.summary().tables[1]); print(f"R²={m2.rsquared:.3f}  R² hiệu chỉnh={m2.rsquared_adj:.3f}  F p={m2.f_pvalue:.2g}")

In [ ]:
# Hệ số chuẩn hoá (Beta trong SPSS) để so sánh độ mạnh giữa biến
z = df[["math", "study_hours", "interest", "anxiety"]].apply(lambda c: (c - c.mean()) / c.std())
print(smf.ols("math ~ study_hours + interest + anxiety", z).fit().params.round(3))

## 3. Kiểm tra giả định & đa cộng tuyến

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
X = df[["study_hours", "interest", "anxiety"]].assign(const=1)
print({c: round(vif(X.values, i), 2) for i, c in enumerate(X.columns) if c != "const"}, " (VIF>5–10 là đáng lo)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.scatterplot(x=m2.fittedvalues, y=m2.resid, ax=ax[0]); ax[0].axhline(0, color="k"); ax[0].set_title("Phần dư vs giá trị dự đoán")
import statsmodels.api as sm; sm.qqplot(m2.resid, line="s", ax=ax[1]); plt.show()

In [ ]:
# Cố tình gây đa cộng tuyến: thêm một biến gần như trùng study_hours
d2 = df.assign(study_min=df.study_hours * 60 + np.random.default_rng(0).normal(0, 20, len(df)))
mm = smf.ols("math ~ study_hours + study_min", d2).fit(); print(mm.summary().tables[1])

**❓** Điều gì xảy ra với sai số chuẩn của hai hệ số? Vì sao mô hình vẫn dự đoán tốt nhưng hệ số không đáng tin?

## 4. Phân tích nhân tố khám phá (EFA) — 6 câu hỏi Likert có gộp thành 2 thang đo không?
SPSS: `Analyze > Dimension Reduction > Factor` (Extraction: Principal axis/ML, Rotation: Varimax).

In [ ]:
items = df[["h1", "h2", "h3", "a1", "a2", "a3"]].dropna().astype(float)
R = np.corrcoef(items.T.values); n_, p_ = items.shape
# Bartlett: các biến có tương quan với nhau không (H0: ma trận tương quan = ma trận đơn vị)
chi2 = -(n_ - 1 - (2 * p_ + 5) / 6) * np.log(np.linalg.det(R)); dfb = p_ * (p_ - 1) / 2
print(f"Bartlett χ²={chi2:.1f}, df={dfb:.0f}, p={stats.chi2.sf(chi2, dfb):.3g}")
# KMO: tương quan riêng phần nhỏ so với tương quan thường -> dữ liệu 'gộp được'
inv = np.linalg.inv(R); pc = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv))); off = ~np.eye(p_, dtype=bool)
print("KMO tổng:", round((R[off] ** 2).sum() / ((R[off] ** 2).sum() + (pc[off] ** 2).sum()), 3), "(>0.6 mới nên làm EFA)")
w, V = np.linalg.eigh(R); idx = np.argsort(w)[::-1]; w, V = w[idx], V[:, idx]
plt.plot(range(1, p_ + 1), w, "o-"); plt.axhline(1, ls="--"); plt.title("Scree plot: giữ nhân tố có eigenvalue > 1"); plt.show()
print("Eigenvalues:", w.round(2))

In [ ]:
def varimax(L, iters=100, tol=1e-8):
    p, k = L.shape; Rm = np.eye(k); d = 0
    for _ in range(iters):
        Lr = L @ Rm
        u, s, vt = np.linalg.svd(L.T @ (Lr ** 3 - Lr @ np.diag((Lr ** 2).sum(0)) / p)); Rm = u @ vt
        if s.sum() < d * (1 + tol): break
        d = s.sum()
    return L @ Rm
L = V[:, :2] * np.sqrt(w[:2])                      # tải nhân tố (thành phần chính) trước xoay
print(pd.DataFrame(varimax(L), index=items.columns, columns=["F1", "F2"]).round(2))

In [ ]:
def cronbach(d): k = d.shape[1]; return k / (k - 1) * (1 - d.var(ddof=1).sum() / d.sum(axis=1).var(ddof=1))
print("Alpha hứng thú:", round(cronbach(items[["h1", "h2", "h3"]]), 3), "| Alpha lo âu:", round(cronbach(items[["a1", "a2", "a3"]]), 3))

**❓** Dữ liệu này được *sinh ra* từ 2 nhân tố ẩn. EFA có tìm lại đúng cấu trúc không? Nếu cấu trúc thật là 3 nhân tố mà bạn ép chọn 2 thì loadings trông thế nào?

## 5. Bài tập
1. Thêm `method` và `gender` vào mô hình `math ~ ...`; phần nào của mô hình đổi? (gợi ý: biến giả/dummy)
2. Viết đoạn "Kết quả" kiểu báo cáo: R², F, hệ số có ý nghĩa, ghi rõ *không* nhân quả.
3. Dùng `predict` dự đoán điểm toán cho em: 6 giờ học, hứng thú 4, lo âu 2, trường B — kèm khoảng dự đoán 95%.